In [ ]:
import pandas as pd

# load dataset :
df = pd.read_csv("../data/Spotify_Tracks.csv")

# Basic sanity check :
df.head()

In [ ]:
# shape of data :
df.shape

In [ ]:
# columns name :
df.columns

In [ ]:
# data types & missing values :
df.info()

In [ ]:
# quick stats :
df.describe()

In [ ]:
# TO CHECK DATA QUALITY :
# === missing values per columns ===
df.isna().sum().sort_values(ascending=False)

In [ ]:
# DUPILCATED ROWS :
df.duplicated().sum()

In [ ]:
#  POPULARITY sanity check :
df["popularity"].describe()

df["popularity"].value_counts().head(10)

In [ ]:
#==== SEGMENT A : EMERGING / ACTIVE TRACKS 
active_df = df[df["popularity"]>0]
active_df.shape

# comparing popularity distribution
active_df["popularity"].describe()



In [ ]:
# STEP 1 : Deriving a New column - POPULARITY LEVEL

# CORE ANALYSIS | QUES : Do audio features systematically changes as popularity increases ?
# popularity bucket :
def popularity_bucket(p):
    if p < 30 :
        return "Low" 
    elif p < 60 :
        return "Medium"
    else :
        return "High"

active_df["popularity_level"] = active_df["popularity"].apply(popularity_bucket)

active_df["popularity_level"].value_counts()

In [ ]:
# STEP 2 : COMPARING AUDIO FEATURES ACROSS POPULARITY LEVELS
audio_features = [
    "danceability","energy","acousticness","speechiness","valence","tempo","loudness"
]
active_df.groupby("popularity_level")[audio_features].mean()

REANCHORING OUR GOAL :
QUES : Are there Medium-POP tracks that already look like High-POP tracks ? 
Those will be our EMERGING TRENDING TRACKS

In [ ]:
# HERE WE ARE GONNA CHECK WHAT MAKES A TRACK HIGH POP :
high_profile = (active_df[active_df["popularity_level"]=="High"]
                [["energy","danceability","acousticness"]].mean())

print("BENCHMARK FOR A HIGH POP TRACK :")
high_profile


In [44]:
# NOW ISOLATING MEDIUM POP TRACKS :

medium_df = active_df[active_df["popularity_level"]=="Medium"]
medium_df.shape

(135932, 19)

DEFINING WHAT AN "EMERGING" TRACKS IS :
ENERGY >= AVG. HIGH ENERGY 
DANCEABILITY >= AVG. HIGH DANCEABILITY
ACOUSTICNESS <= AVG. HIGH ACOUSTICNESS

In [43]:
emerging_df = medium_df[
    (medium_df["energy"] >= high_profile["energy"]) &
    (medium_df["danceability"] >= high_profile["danceability"]) &
    (medium_df["acousticness"] <= high_profile["acousticness"]) 
]
emerging_df.shape

(18903, 19)

In [42]:
# INSPECTING WHAT WE HAVE FOUND :
emerging_df[
    ["track_name", "artist_name", "popularity", 
     "energy", "danceability", "acousticness"]
].head(10)


,track_name,artist_name,popularity,energy,danceability,acousticness
701,King of the Pines (feat. Upchurch),Tommy Chayne,44,0.808,0.734,0.01810
703,Sugar Daddy,Pistol Annies,43,0.870,0.647,0.00761
718,About To,Adam Sanders,42,0.668,0.635,0.02710
740,Boy Gets A Truck,Keith Urban,44,0.693,0.648,0.00667
751,Back That Thing Up,Justin Moore,41,0.896,0.679,0.04740
755,Let It Ride,Bachman-Turner Overdrive,52,0.781,0.632,0.15700
765,One Way Ticket (Because I Can),LeAnn Rimes,44,0.778,0.696,0.02910
766,Where I Come From (Remix) [feat. Colt Ford & T...,Montgomery Gentry,42,0.750,0.756,0.03890
772,"Chicks, Trucks, and Beer - feat. Colt Ford",Tyler Farr,43,0.915,0.637,0.01250
774,The Wind - Greatest Hits Version,Zac Brown Band,44,0.948,0.691,0.20500


In [50]:
# HERE ARE GONNA FIND OUT WHAT % OF TRACKS ARE EMERGING :
percent_emerging=len(emerging_df)/len(medium_df) * 100

print(f"OUT OF ~135000 TRACKS ONLY {percent_emerging} % TRACKS SHOW SOUND CHARACTERISTIC OF HIGH POP TRACKS")




OUT OF ~135000 TRACKS ONLY 13.906217814789748 % TRACKS SHOW SOUND CHARACTERISTIC OF HIGH POP TRACKS
